<a href="https://colab.research.google.com/github/thach-phu/python-en-vi-machine-translation/blob/main/notebooks/baseline_opus_mt.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import torch

print("=== KIỂM TRA GPU ===")
if torch.cuda.is_available():
    device_name = torch.cuda.get_device_name(0)
    gpu_count = torch.cuda.device_count()
    print(f" GPU khả dụng: {device_name}")
    print(f" Số lượng GPU: {gpu_count}")
    print(f" Bộ nhớ GPU: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
else:
    print(" Không tìm thấy GPU. Hãy bật GPU trong môi trường:")
    print("   - Colab: Runtime > Change runtime type > Hardware accelerator > T4 GPU")
    print("   - Kaggle: Notebook options > Accelerator > GPU T4 x2 / P100")

=== KIỂM TRA GPU ===
 GPU khả dụng: Tesla T4
 Số lượng GPU: 1
 Bộ nhớ GPU: 15.64 GB


In [2]:
# Cài đặt các thư viện Hugging Face và thư viện tính điểm SacreBLEU
!pip install -q transformers datasets sacrebleu evaluate sentencepiece

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 129.6/129.6 kB 5.5 MB/s eta 0:00:00


In [4]:
import torch
import transformers
from datasets import load_dataset
import sacrebleu
import evaluate

print("=== KIỂM TRA PHIÊN BẢN THƯ VIỆN ===")
print(f"PyTorch version: {torch.__version__}")
print(f"Transformers version: {transformers.__version__}")

# 1. Kiểm tra SacreBLEU với dữ liệu mẫu
preds = ["I am coding in a notebook."]
refs = [["I am coding in a notebook.", "I am writing code in a notebook."]]

bleu = sacrebleu.corpus_bleu(preds, refs)
print(f"\n✅ SacreBLEU score test: {bleu.score:.2f}")

# 2. Tải thử dataset với namespace đầy đủ
print("\n🔄 Đang thử tải dataset mẫu từ Hugging Face...")
dataset = load_dataset("cornell-movie-review-data/rotten_tomatoes", split="train[:5]")
print(f"✅ Load dataset thành công! Số lượng dòng thử nghiệm: {len(dataset)}")
print("Mẫu dữ liệu:", dataset[0])

=== KIỂM TRA PHIÊN BẢN THƯ VIỆN ===
PyTorch version: 2.11.0+cu128
Transformers version: 5.16.1

✅ SacreBLEU score test: 100.00

🔄 Đang thử tải dataset mẫu từ Hugging Face...


README.md:   0%|          | 0.00/7.46k [00:00<?, ?B/s]

train.parquet: reconstructing file:   0%|          |  0.00B /  699kB            

train.parquet: downloading bytes:           |  0.00B            

validation.parquet: reconstructing file:   0%|          |  0.00B / 90.0kB            

...

test.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/8530 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/1066 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1066 [00:00<?, ? examples/s]

✅ Load dataset thành công! Số lượng dòng thử nghiệm: 5
Mẫu dữ liệu: {'text': 'the rock is destined to be the 21st century\'s new " conan " and that he\'s going to make a splash even greater than arnold schwarzenegger , jean-claud van damme or steven segal .', 'label': 1}


In [5]:
import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

# 1. Khởi tạo tên mô hình dịch Anh - Việt chuẩn từ Helsinki-NLP
model_name = "Helsinki-NLP/opus-mt-en-vi"

print("🔄 Đang tải mô hình và tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

# Chuyển mô hình sang GPU nếu có
device = "cuda" if torch.cuda.is_available() else "cpu"
model = model.to(device)
print(f"✅ Đã tải xong! Thiết bị đang sử dụng: {device}\n")

# 2. Danh sách 5-10 câu tiếng Anh thử nghiệm
english_sentences = [
    "Artificial Intelligence is transforming the world.",
    "Machine translation helps bridge the language gap between cultures.",
    "Learning Python programming is very useful for data science.",
    "Google Colab provides free GPU access for training machine learning models.",
    "We are building an English to Vietnamese translation project.",
    "Natural Language Processing has improved significantly in recent years."
]

print("=== KẾT QUẢ DỊCH MÁY ANH - VIỆT (opus-mt-en-vi) ===\n")

for i, src_text in enumerate(english_sentences, 1):
    # Mã hóa văn bản đầu vào
    inputs = tokenizer(src_text, return_tensors="pt", padding=True, truncation=True).to(device)

    # Thực hiện dịch (generation)
    with torch.no_grad():
        translated_tokens = model.generate(**inputs, max_length=128)

    # Giải mã kết quả về dạng văn bản
    translated_text = tokenizer.decode(translated_tokens[0], skip_special_tokens=True)

    print(f"[{i}] 🇬🇧 En: {src_text}")
    print(f"    🇻🇳 Vi: {translated_text}\n")

🔄 Đang tải mô hình và tokenizer...


config.json:   0%|          | 0.00/1.39k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/44.0 [00:00<?, ?B/s]

source.spm:   0%|          | 0.00/809k [00:00<?, ?B/s]

target.spm:   0%|          | 0.00/756k [00:00<?, ?B/s]

...

Loading weights:   0%|          | 0/258 [00:00<?, ?it/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  289MB            

model.safetensors: downloading bytes:           |  0.00B            

generation_config.json:   0%|          | 0.00/293 [00:00<?, ?B/s]

✅ Đã tải xong! Thiết bị đang sử dụng: cuda

=== KẾT QUẢ DỊCH MÁY ANH - VIỆT (opus-mt-en-vi) ===

[1] 🇬🇧 En: Artificial Intelligence is transforming the world.
    🇻🇳 Vi: Tình dục là chuyện bình thường trên thế giới.

[2] 🇬🇧 En: Machine translation helps bridge the language gap between cultures.
    🇻🇳 Vi: Máy dịch thuật giúp nối các khoảng cách ngôn ngữ giữa các nền văn hóa.

[3] 🇬🇧 En: Learning Python programming is very useful for data science.
    🇻🇳 Vi: Chương trình mô phỏng lập trình là rất hữu ích cho dữ liệu khoa học.

[4] 🇬🇧 En: Google Colab provides free GPU access for training machine learning models.
    🇻🇳 Vi: Google Colab cung cấp tự do GPU quyền truy cập để đào tạo máy học mô hình.

[5] 🇬🇧 En: We are building an English to Vietnamese translation project.
    🇻🇳 Vi: Chúng tôi đang xây dựng một công trình dịch thuật bằng tiếng Anh.

[6] 🇬🇧 En: Natural Language Processing has improved significantly in recent years.
    🇻🇳 Vi: Trong những năm gần đây, việc phân chia ngôn ngữ 